In [59]:
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, RocCurveDisplay, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, precision_recall_curve, roc_curve
import plotly.express as px
import matplotlib.pyplot as plt
import bioframe as bf

pd.options.display.max_columns = 100

# Functions

In [46]:
def training(df, features, test_chrom):
    """ Trains a model returns classifications"""

    X_train = df.loc[~df['chrom'].isin(test_chrom), features]
    X_test = df.loc[df['chrom'].isin(test_chrom), features]
    y_train = df.loc[~df['chrom'].isin(test_chrom), 'confirmed']
    y_test = df.loc[df['chrom'].isin(test_chrom), 'confirmed']

    RF = RandomForestClassifier(n_estimators=100)
    RF.fit(X_train, y_train)

    predictions = RF.predict(X_test)
    proba = RF.predict_proba(X_test)
    X_test['confirmed'] = y_test
    X_test['predictions'] = predictions
    X_test['probability'] = proba[:, 1]
    result = df[['id', 'sample', 'method', 'type', 'chrom', 'start', 'end']].merge(X_test[['confirmed', 'predictions', 'probability']], left_index=True, right_index=True)

    return result

In [21]:
def load_sample_data(SAMPLE, REF):
    """ Load sample data from a single sample. """
    
    FEATURE_DIR = f'/confidential/FamilyR13/DATA/10x/sv_compare/results/{SAMPLE}_{REF}/ensemble'
    df_raw = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.raw.tsv', sep='\t')
    df_ref = pd.read_csv(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.ref.tsv', sep='\t', low_memory=False)

    filenames_aln_ill = glob(f'{FEATURE_DIR}/{SAMPLE}_{REF}.SVs.aln.ill.*.tsv')
    df_aln_ill = pd.concat([pd.read_csv(f, sep='\t') for f in filenames_aln_ill], ignore_index=True)

    df = df_raw.merge(df_ref.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='left')
    df = df.merge(df_aln_ill.drop(['sample', 'type', 'chrom', 'chrom2', 'start', 'end'], axis=1), on='id', how='inner')

    return df

In [56]:
# Mark variants which have been found by multiple callers

def extract_overlap_ids(df1, df2):
    """ Extracts SV IDs of overlapping variants"""

    closest_intervals = bf.closest(df1, df2, suffixes=('_1','_2'))
    closest_intervals['diff_start'] = abs(closest_intervals['start_1'] - closest_intervals['start_2'])
    closest_intervals['diff_end'] = abs(closest_intervals['end_1'] - closest_intervals['end_2'])
    closest_intervals['diff_size'] = closest_intervals.apply(lambda x: min([x['size_1'], x['size_2']]) / max([x['size_1'], x['size_2']]), axis=1)
    overlapping_svs = closest_intervals[(closest_intervals['diff_start'] < 50) & (closest_intervals['diff_end'] < 50) & (closest_intervals['diff_size'] > 0.7)].copy()
    
    return overlapping_svs[['id_1', 'id_2']].reset_index(drop=True)

# Script

In [22]:
# PARAMETERS
SAMPLES = ['17-08']
REF = 'hg38'
TYPE = 'DEL'
CHROMS = ['chr1', 'chr2', 'chr3', 'chr4', 'chr5', 'chr6', 'chr7', 'chr8', 'chr9', 'chr10', 'chr11', 'chr12', 'chr13', 
          'chr14', 'chr15', 'chr16', 'chr17', 'chr18', 'chr19', 'chr20', 'chr21', 'chr22', 'chrX']

In [23]:
# Load Data
dfs = []
for SAMPLE in SAMPLES:
    df = load_sample_data(SAMPLE, REF)
    dfs.append(df)

df = pd.concat(dfs, ignore_index=True)

In [24]:
# Filter NaNs
features = ['size'] + list(df.columns[13:])
df = df.dropna(subset=features).copy().reset_index(drop=True)

# Select SV Type
df = df[df['type'] == 'DEL'].copy().reset_index(drop=True)

## Create set for manual curation

In [ ]:
fp_svs = set()
fn_svs = set()
for chrom in tqdm(CHROMS):
    result = training(df, features, [chrom])
    curr_fp_svs = list(result.loc[(result['confirmed'] == 0) & (result['predictions'] == 1), 'id'])
    curr_fp_svs = list(result.loc[(result['confirmed'] == 1) & (result['predictions'] == 0), 'id'])
    fp_svs.update(curr_fp_svs)
    fn_svs.update(curr_fn_svs)

# Evaluate Classifier

In [47]:
chrom_train = ['chr1', 'chr3', 'chr4', 'chr5', 'chr7', 'chr8', 'chr10', 'chr11', 'chr12', 'chr13', 'chr14', 'chr15', 'chr16', 'chr17', 'chr19', 'chr20', 'chr21']
chrom_test = ['chr2', 'chr6', 'chr9', 'chr18', 'chr22']

In [48]:
result = training(df, features, chrom_test)
result = result.merge(df[['id', 'qual']], on='id', how='left')

In [80]:
method_dfs = dict()
methods = list(result['method'].unique())
for method in methods:
    result['qual_' + method] = 0
    result.loc[result['method'] == method, 'qual_' + method] = result.loc[result['method'] == method, 'qual']
    method_dfs[method] = df.loc[df['method'] == method, ['id', 'method', 'chrom', 'start', 'end', 'type', 'size']].copy()
result.drop('qual', axis=1, inplace=True)

In [115]:
# check if SVs are found by multiple callers
for i in range(len(methods)):
    for j in range(i+1, len(methods)):
        overlap_ids = extract_overlap_ids(method_dfs[methods[i]], method_dfs[methods[j]])
        for l in range(len(overlap_ids)):
            if overlap_ids.loc[l, 'id_1'] in result['id'].to_list():
                result.loc[(result['id'] == overlap_ids.loc[l, 'id_1']), 'qual_' + methods[i]] = result.loc[(result['id'] == overlap_ids.loc[l, 'id_2']), 'qual_' + methods[j]].values[0]





                #print(f'{overlap_ids.loc[l, "id_1"]} is found by {methods[i]} and {methods[j]}')
        
        

In [118]:
result.loc[(result['id'] == overlap_ids.loc[l, 'id_1']), 'qual_' + methods[i]] = result.loc[(result['id'] == overlap_ids.loc[l, 'id_2']), 'qual_' + methods[j]].values[0]

IndexError: index 0 is out of bounds for axis 0 with size 0

In [116]:
result[(result['qual_delly'] > 0) & (result['qual_manta'] > 0)]

,id,sample,method,type,chrom,start,end,confirmed,predictions,probability,qual_delly,qual_manta,qual_lumpy


In [98]:
overlap_ids.loc[l, 'id_1']

'MantaDEL:171004:0:1:0:0:0'

In [88]:
result[(result['method'] == 'manta') & (result['id'] == 'MantaDEL:110:0:0:0:0:0'), 'qual_manta'] = 

,id,sample,method,type,chrom,start,end,confirmed,predictions,probability,qual_delly,qual_manta,qual_lumpy


In [86]:
result

,id,sample,method,type,chrom,start,end,confirmed,predictions,probability,qual_delly,qual_manta,qual_lumpy
0,DEL00005848,17-08,delly,DEL,chr2,147791,147822,0,0,0.04,1200,0,0.000000
1,DEL00005849,17-08,delly,DEL,chr2,160744,238980999,0,0,0.01,9,0,0.000000
2,DEL00005851,17-08,delly,DEL,chr2,207389,32916225,0,0,0.09,443,0,0.000000
3,DEL00005857,17-08,delly,DEL,chr2,348407,349028,0,0,0.08,120,0,0.000000
4,DEL00005861,17-08,delly,DEL,chr2,732048,32916228,0,0,0.14,144,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5186,4124,17-08,lumpy,DEL,chr22,49226575,49228371,0,1,0.52,0,0,551.179993
5187,4126,17-08,lumpy,DEL,chr22,49384302,49386135,0,0,0.13,0,0,50.779999
5188,4127,17-08,lumpy,DEL,chr22,49407555,49407910,1,1,0.51,0,0,476.609985
5189,4128,17-08,lumpy,DEL,chr22,49667596,49668192,0,1,0.53,0,0,110.199997
